In [1]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Vivek_Vihar_Delhi_DPCC_2023.xlsx")

In [4]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,309.0,191.0,217.0,89.0,83.0,104.0,69.0,91.0,141.0,NaN,350.0,428.0
1,2,386.0,254.0,221.0,135.0,74.0,117.0,68.0,98.0,140.0,159.0,385.0,374.0
2,3,410.0,251.0,147.0,219.0,117.0,161.0,110.0,82.0,137.0,154.0,442.0,350.0
3,4,365.0,280.0,133.0,114.0,117.0,196.0,150.0,92.0,138.0,183.0,415.0,339.0
4,5,396.0,272.0,NaN,125.0,172.0,158.0,105.0,93.0,110.0,193.0,445.0,335.0
5,6,421.0,307.0,140.0,164.0,221.0,125.0,NaN,97.0,106.0,219.0,433.0,308.0
6,7,415.0,324.0,148.0,146.0,171.0,201.0,NaN,103.0,95.0,248.0,408.0,389.0
7,8,397.0,155.0,231.0,160.0,136.0,179.0,87.0,117.0,92.0,171.0,NaN,363.0
8,9,449.0,211.0,105.0,221.0,201.0,175.0,71.0,120.0,43.0,202.0,437.0,366.0
9,10,429.0,194.0,157.0,215.0,205.0,144.0,NaN,127.0,34.0,NaN,288.0,335.0


In [5]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    36 non-null     float64
 2   February   32 non-null     float64
 3   March      34 non-null     float64
 4   April      34 non-null     float64
 5   May        36 non-null     float64
 6   June       34 non-null     float64
 7   July       18 non-null     float64
 8   August     34 non-null     float64
 9   September  32 non-null     float64
 10  October    34 non-null     float64
 11  November   34 non-null     float64
 12  December   34 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


In [6]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [7]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [8]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [10]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,309.0,191.00000,217.000000,89.000000,83.0,104.000000,70.388889,91.000000,141.00000,186.558824,350.000000,428.000000
1,2,386.0,254.00000,221.000000,135.000000,74.0,117.000000,70.388889,98.000000,140.00000,159.000000,385.000000,374.000000
2,3,410.0,251.00000,147.000000,219.000000,117.0,161.000000,70.388889,82.000000,137.00000,154.000000,442.000000,350.000000
3,4,365.0,280.00000,133.000000,114.000000,117.0,196.000000,70.388889,92.000000,138.00000,183.000000,415.000000,339.000000
4,5,396.0,272.00000,161.705882,125.000000,172.0,158.000000,70.388889,93.000000,110.00000,193.000000,445.000000,335.000000
5,6,421.0,307.00000,140.000000,164.000000,221.0,125.000000,70.388889,97.000000,106.00000,219.000000,433.000000,308.000000
6,7,415.0,324.00000,148.000000,146.000000,171.0,201.000000,70.388889,103.000000,95.00000,248.000000,408.000000,389.000000
7,8,397.0,155.00000,231.000000,160.000000,136.0,179.000000,70.388889,117.000000,92.00000,171.000000,302.147059,363.000000
8,9,449.0,211.00000,105.000000,221.000000,201.0,175.000000,70.388889,120.000000,43.00000,202.000000,437.000000,366.000000
9,10,429.0,194.00000,157.000000,215.000000,205.0,144.000000,70.388889,127.000000,93.90625,186.558824,288.000000,335.000000
